In [1]:
import os
import json
import torch
from PIL import Image
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText
from tqdm import tqdm
from collections import Counter

# Confirm device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

Device: mps
PyTorch: 2.12.0


In [2]:
MODEL_ID = "google/medgemma-4b-it"

print(f"Loading processor for {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f"Loading model...")
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)
model = model.to(device)
model.eval()

print(f"Model loaded. Device: {next(model.parameters()).device}")

Loading processor for google/medgemma-4b-it...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded. Device: mps:0


In [3]:
SLAKE_IMGS_DIR = os.path.expanduser('~/vlm_benchmark/data/slake_imgs/imgs')

def to_rgb(img: Image.Image) -> Image.Image:
    if img.mode == 'RGB':
        return img
    return img.convert('RGB')

def load_slake_image(img_name: str) -> Image.Image:
    path = os.path.join(SLAKE_IMGS_DIR, img_name)
    return Image.open(path).convert('RGB')

def build_prompt(question: str, is_closed: bool) -> str:
    """
    MultiMedEval protocol:
    - Closed-ended: prepend 'Answer the question with yes or no.'
    - Open-ended: question as-is
    Prompt is: <image>\n{question}
    The image token is handled by the processor; we just pass the text.
    """
    if is_closed:
        return f"Answer the question with yes or no. {question}"
    return question

print("Utilities defined.")

Utilities defined.


In [4]:
def run_inference(image: Image.Image, question: str, is_closed: bool) -> str:
    prompt_text = build_prompt(question, is_closed)
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": prompt_text},
            ]
        }
    ]
    
    # Apply chat template
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = processor(
        text=text,
        images=image,
        return_tensors="pt"
    ).to(device)
    
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
        )
    
    # Decode only the newly generated tokens
    input_len = inputs["input_ids"].shape[-1]
    generated = output_ids[0][input_len:]
    decoded = processor.decode(generated, skip_special_tokens=True).strip()
    
    return decoded

print("Inference function defined.")

Inference function defined.


In [5]:
slake = load_dataset('BoKelvin/SLAKE')
slake_en_test = slake['test'].filter(lambda x: x['q_lang'] == 'en')

print("=== SLAKE 5-sample smoke test ===\n")
for i, sample in enumerate(slake_en_test.select(range(5))):
    is_closed = sample['answer_type'] == 'CLOSED'
    image     = load_slake_image(sample['img_name'])
    prediction = run_inference(image, sample['question'], is_closed)
    
    print(f"[{i+1}] Type: {'CLOSED' if is_closed else 'OPEN'}")
    print(f"     Q:    {sample['question']}")
    print(f"     GT:   {sample['answer']}")
    print(f"     Pred: {prediction}")
    print()

=== SLAKE 5-sample smoke test ===

[1] Type: OPEN
     Q:    What modality is used to take this image?
     GT:   CT
     Pred: Based on the image characteristics, the modality used to take this image is **Computed Tomography (CT)**.

Here's why:

*   **Cross-sectional view:** CT scans provide cross-sectional images of the body, allowing visualization of internal structures.
*   **Bone detail:** The image shows good visualization of bone structures, which is characteristic of CT.
*   **Density differences:** Different tissues appear in different shades of gray based on their density, a hallmark of CT

[2] Type: OPEN
     Q:    Which part of the body does this image belong to?
     GT:   Chest
     Pred: Based on the image, it appears to be a cross-sectional CT scan of the **chest**. You can see the lungs, heart, major blood vessels, and bones of the rib cage.

[3] Type: OPEN
     Q:    What is the main organ in the image?
     GT:   Lung, Spinal Cord
     Pred: The main organ in the im

In [6]:
  def build_prompt(question: str, is_closed: bool) -> str:
    """
    MultiMedEval protocol with conciseness instruction to suppress
    markdown formatting and verbose explanations.
    """
    if is_closed:
        return f"Answer the question with yes or no. {question}"
    return f"{question} Answer concisely in as few words as possible."

print("Updated prompt builder defined.")

Updated prompt builder defined.


In [7]:
print("=== SLAKE 5-sample smoke test (concise prompt) ===\n")
for i, sample in enumerate(slake_en_test.select(range(5))):
    is_closed  = sample['answer_type'] == 'CLOSED'
    image      = load_slake_image(sample['img_name'])
    prediction = run_inference(image, sample['question'], is_closed)
    
    print(f"[{i+1}] Type: {'CLOSED' if is_closed else 'OPEN'}")
    print(f"     Q:    {sample['question']}")
    print(f"     GT:   {sample['answer']}")
    print(f"     Pred: {prediction}")
    print()

=== SLAKE 5-sample smoke test (concise prompt) ===

[1] Type: OPEN
     Q:    What modality is used to take this image?
     GT:   CT
     Pred: Computed tomography (CT)

[2] Type: OPEN
     Q:    Which part of the body does this image belong to?
     GT:   Chest
     Pred: Chest

[3] Type: OPEN
     Q:    What is the main organ in the image?
     GT:   Lung, Spinal Cord
     Pred: Lung

[4] Type: OPEN
     Q:    What is the largest organ in the picture?
     GT:   Lung
     Pred: Lung

[5] Type: CLOSED
     Q:    Does the picture contain liver?
     GT:   No
     Pred: No



In [8]:
# VQA-RAD 5-sample smoke test
vqarad = load_dataset('flaviagiammarino/vqa-rad')
vqarad_test = vqarad['test']

def is_closed_vqarad(answer: str) -> bool:
    return answer.strip().lower() in ('yes', 'no')

print("=== VQA-RAD 5-sample smoke test ===\n")
for i, sample in enumerate(vqarad_test.select(range(5))):
    is_closed  = is_closed_vqarad(sample['answer'])
    image      = to_rgb(sample['image'])
    prediction = run_inference(image, sample['question'], is_closed)
    
    print(f"[{i+1}] Type: {'CLOSED' if is_closed else 'OPEN'}")
    print(f"     Q:    {sample['question']}")
    print(f"     GT:   {sample['answer']}")
    print(f"     Pred: {prediction}")
    print()

=== VQA-RAD 5-sample smoke test ===

[1] Type: CLOSED
     Q:    is there evidence of an aortic aneurysm?
     GT:   yes
     Pred: No.

[2] Type: CLOSED
     Q:    is there airspace consolidation on the left side?
     GT:   yes
     Pred: Yes

[3] Type: CLOSED
     Q:    is there any intraparenchymal abnormalities in the lung fields?
     GT:   no
     Pred: No.

[4] Type: OPEN
     Q:    which side of the heart border is obscured?
     GT:   right
     Pred: Right heart border.

[5] Type: OPEN
     Q:    where are the kidney?
     GT:   not seen here
     Pred: The kidneys are not visible in this image. This is a cross-sectional view of the upper abdomen, showing organs like the liver, spleen, and stomach.



In [9]:
# PathVQA 5-sample smoke test
pvqa = load_dataset('flaviagiammarino/path-vqa')
pvqa_test = pvqa['test']

def is_closed_pvqa(answer: str) -> bool:
    return answer.strip().lower() in ('yes', 'no')

print("=== PathVQA 5-sample smoke test ===\n")
for i, sample in enumerate(pvqa_test.select(range(5))):
    is_closed  = is_closed_pvqa(sample['answer'])
    image      = to_rgb(sample['image'])
    prediction = run_inference(image, sample['question'], is_closed)
    
    print(f"[{i+1}] Type: {'CLOSED' if is_closed else 'OPEN'}")
    print(f"     Q:    {sample['question']}")
    print(f"     GT:   {sample['answer']}")
    print(f"     Pred: {prediction}")
    print()

=== PathVQA 5-sample smoke test ===

[1] Type: OPEN
     Q:    what are positively charged, thus allowing the compaction of the negatively charged dna?
     GT:   the histone subunits
     Pred: Histones

[2] Type: OPEN
     Q:    how are the histone subunits charged?
     GT:   positively charged
     Pred: Histone subunits are positively charged.

[3] Type: OPEN
     Q:    what is showing increased eosinophilia of cytoplasm, and swelling of occasional cells?
     GT:   early (reversible) ischemic injury
     Pred: The image shows increased eosinophilia of cytoplasm and swelling of occasional cells, which could indicate **cellular damage or stress**.

[4] Type: CLOSED
     Q:    does mycobacterium avium infection in a duodenal biopsy from a patient with aids show massive intracellular macrophage infection with acid-fast organisms filamentous and pink in this acid-fast stain preparation?
     GT:   yes
     Pred: Yes

[5] Type: OPEN
     Q:    what shows branching papillae having flbro

In [10]:
# Answer extraction and scoring functions

import re
import string
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── Answer cleaning ────────────────────────────────────────────────────

def clean_answer(text: str) -> str:
    """
    Remove markdown formatting, strip punctuation, lowercase.
    Used before tokenization for F1/accuracy scoring.
    """
    # Remove markdown bold/italic
    text = re.sub(r'\*+', '', text)
    # Take only the first sentence to cut verbose explanations
    text = re.split(r'(?<=[.!?])\s', text)[0]
    # Strip leading/trailing whitespace
    text = text.strip()
    # Remove trailing punctuation
    text = text.rstrip(string.punctuation)
    return text.lower()

def extract_closed_answer(text: str) -> str:
    """
    For closed-ended questions extract yes/no from the first word.
    """
    cleaned = clean_answer(text)
    tokens  = cleaned.split()
    if not tokens:
        return ''
    first = tokens[0].strip(string.punctuation)
    if first in ('yes', 'no'):
        return first
    # Fallback: search anywhere in the text
    for token in tokens:
        t = token.strip(string.punctuation)
        if t in ('yes', 'no'):
            return t
    return cleaned  # Return as-is if no yes/no found

# ── Token-level F1 (MultiMedEval protocol) ────────────────────────────

def tokenize_answer(text: str) -> list[str]:
    """Lowercase, remove punctuation, tokenize."""
    text  = re.sub(r'\*+', '', text).lower()
    text  = text.translate(str.maketrans('', '', string.punctuation))
    return word_tokenize(text)

def token_f1(prediction: str, ground_truth: str) -> dict:
    pred_tokens = tokenize_answer(prediction)
    gt_tokens   = tokenize_answer(ground_truth)
    
    if not pred_tokens or not gt_tokens:
        return {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
    
    pred_set = Counter(pred_tokens)
    gt_set   = Counter(gt_tokens)
    
    common       = sum((pred_set & gt_set).values())
    precision    = common / len(pred_tokens) if pred_tokens else 0.0
    recall       = common / len(gt_tokens)   if gt_tokens   else 0.0
    f1           = (2 * precision * recall / (precision + recall)
                    if (precision + recall) > 0 else 0.0)
    
    return {'f1': f1, 'precision': precision, 'recall': recall}

# ── Accuracy thresholds (MultiMedEval protocol) ───────────────────────

def is_correct(prediction: str, ground_truth: str, is_closed: bool) -> bool:
    """
    Closed-ended: recall >= 0.5
    Open-ended:   recall >= 0.75
    """
    scores    = token_f1(prediction, ground_truth)
    threshold = 0.5 if is_closed else 0.75
    return scores['recall'] >= threshold

# ── BLEU (Bilingual Evaluation Understudy) (non-tokenized, sacrebleu) ───────────────────────────────────

from sacrebleu.metrics import BLEU

bleu_metric = BLEU(effective_order=True)

def compute_bleu(prediction: str, ground_truth: str) -> float:
    score = bleu_metric.sentence_score(
        hypothesis=prediction.lower(),
        references=[ground_truth.lower()]
    )
    return score.score  # Returns 0-100

print("Scoring functions defined.")

Scoring functions defined.


In [11]:
# Manually verify scoring logic against our known smoke test outputs
test_cases = [
    # (prediction, ground_truth, is_closed, description)
    ("Computed tomography (CT)", "CT",               False, "SLAKE [1] open"),
    ("Chest",                    "Chest",             False, "SLAKE [2] open - exact"),
    ("Lung",                     "Lung, Spinal Cord", False, "SLAKE [3] open - partial"),
    ("No",                       "yes",               True,  "VQA-RAD [1] closed - wrong"),
    ("Yes",                      "yes",               True,  "VQA-RAD [2] closed - correct"),
    ("Right heart border",       "right",             False, "VQA-RAD [4] open"),
    ("Histones",                 "the histone subunits", False, "PathVQA [1] open"),
    ("Yes",                      "yes",               True,  "PathVQA [4] closed - correct"),
]

print(f"{'Description':<35} {'F1':>6} {'Recall':>8} {'Correct':>8} {'BLEU':>7}")
print("-" * 70)
for pred, gt, closed, desc in test_cases:
    scores  = token_f1(pred, gt)
    correct = is_correct(pred, gt, closed)
    bleu    = compute_bleu(pred, gt)
    print(f"{desc:<35} {scores['f1']:>6.3f} {scores['recall']:>8.3f} "
          f"{'YES' if correct else 'NO':>8} {bleu:>7.2f}")

Description                             F1   Recall  Correct    BLEU
----------------------------------------------------------------------
SLAKE [1] open                       0.500    1.000      YES   10.68
SLAKE [2] open - exact               1.000    1.000      YES  100.00
SLAKE [3] open - partial             0.500    0.333       NO    4.98
VQA-RAD [1] closed - wrong           0.000    0.000       NO    0.00
VQA-RAD [2] closed - correct         1.000    1.000      YES  100.00
VQA-RAD [4] open                     0.500    1.000      YES   27.52
PathVQA [1] open                     0.000    0.000       NO    0.00
PathVQA [4] closed - correct         1.000    1.000      YES  100.00


In [12]:
# Full dataset runner

def run_dataset(
    samples,
    get_image_fn,
    get_question_fn,
    get_answer_fn,
    get_is_closed_fn,
    dataset_name: str,
    model_name: str,
    output_dir: str = os.path.expanduser('~/vlm_benchmark/outputs'),
    max_samples: int = None
) -> str:
    """
    Runs inference on a dataset and saves results to JSONL.
    Returns path to the output file.
    """
    os.makedirs(output_dir, exist_ok=True)
    safe_model = model_name.replace('/', '_')
    out_path   = os.path.join(output_dir, f"{safe_model}__{dataset_name}.jsonl")
    
    if max_samples:
        samples = samples.select(range(max_samples))
    
    results = []
    errors  = 0
    
    for i, sample in enumerate(tqdm(samples, desc=f"{model_name} | {dataset_name}")):
        try:
            image     = get_image_fn(sample)
            question  = get_question_fn(sample)
            answer    = get_answer_fn(sample)
            is_closed = get_is_closed_fn(sample)
            
            prediction = run_inference(image, question, is_closed)
            
            results.append({
                'idx':        i,
                'question':   question,
                'ground_truth': answer,
                'prediction': prediction,
                'is_closed':  is_closed,
                'model':      model_name,
                'dataset':    dataset_name,
            })
        except Exception as e:
            errors += 1
            results.append({
                'idx':        i,
                'question':   get_question_fn(sample) if sample else '',
                'ground_truth': get_answer_fn(sample) if sample else '',
                'prediction': '',
                'is_closed':  False,
                'model':      model_name,
                'dataset':    dataset_name,
                'error':      str(e),
            })
    
    with open(out_path, 'w') as f:
        for r in results:
            f.write(json.dumps(r) + '\n')
    
    print(f"Saved {len(results)} results ({errors} errors) → {out_path}")
    return out_path

print("Runner defined.")

Runner defined.


In [13]:
# Dataset accessor functions

# ── SLAKE ──────────────────────────────────────────────────────────────
slake_en_test = slake['test'].filter(lambda x: x['q_lang'] == 'en')

slake_fns = dict(
    get_image_fn     = lambda s: load_slake_image(s['img_name']),
    get_question_fn  = lambda s: s['question'],
    get_answer_fn    = lambda s: s['answer'],
    get_is_closed_fn = lambda s: s['answer_type'] == 'CLOSED',
)

# ── VQA-RAD ────────────────────────────────────────────────────────────
vqarad_fns = dict(
    get_image_fn     = lambda s: to_rgb(s['image']),
    get_question_fn  = lambda s: s['question'],
    get_answer_fn    = lambda s: s['answer'],
    get_is_closed_fn = lambda s: s['answer'].strip().lower() in ('yes', 'no'),
)

# ── PathVQA ────────────────────────────────────────────────────────────
pvqa_fns = dict(
    get_image_fn     = lambda s: to_rgb(s['image']),
    get_question_fn  = lambda s: s['question'],
    get_answer_fn    = lambda s: s['answer'],
    get_is_closed_fn = lambda s: s['answer'].strip().lower() in ('yes', 'no'),
)

print("Accessors defined.")

Accessors defined.


In [16]:
import re, string, json, os
from nltk.tokenize import word_tokenize
from collections import Counter
from sacrebleu.metrics import BLEU
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

bleu_metric = BLEU(effective_order=True)

def tokenize_answer(text):
    text = re.sub(r'\*+', '', text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return word_tokenize(text)

def token_f1(prediction, ground_truth):
    pred_tokens = tokenize_answer(prediction)
    gt_tokens   = tokenize_answer(ground_truth)
    if not pred_tokens or not gt_tokens:
        return {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}
    pred_set = Counter(pred_tokens)
    gt_set   = Counter(gt_tokens)
    common    = sum((pred_set & gt_set).values())
    precision = common / len(pred_tokens)
    recall    = common / len(gt_tokens)
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return {'f1': f1, 'precision': precision, 'recall': recall}

def is_correct(prediction, ground_truth, is_closed):
    scores    = token_f1(prediction, ground_truth)
    threshold = 0.5 if is_closed else 0.75
    return scores['recall'] >= threshold

def compute_bleu(prediction, ground_truth):
    return bleu_metric.sentence_score(
        hypothesis=prediction.lower(),
        references=[ground_truth.lower()]
    ).score

def score_jsonl(path):
    records = [json.loads(l) for l in open(path)]
    records = [r for r in records if 'error' not in r]
    closed  = [r for r in records if r['is_closed']]
    open_   = [r for r in records if not r['is_closed']]
    def avg_f1(recs):
        if not recs: return 0.0
        return sum(token_f1(r['prediction'], r['ground_truth'])['f1']
                   for r in recs) / len(recs)
    def avg_recall(recs):
        if not recs: return 0.0
        return sum(token_f1(r['prediction'], r['ground_truth'])['recall']
                   for r in recs) / len(recs)
    def accuracy(recs, is_closed):
        if not recs: return 0.0
        return sum(is_correct(r['prediction'], r['ground_truth'], is_closed)
                   for r in recs) / len(recs)
    def avg_bleu(recs):
        if not recs: return 0.0
        return sum(compute_bleu(r['prediction'], r['ground_truth'])
                   for r in recs) / len(recs)
    return {
        'dataset':        os.path.basename(path),
        'n_total':        len(records),
        'n_closed':       len(closed),
        'n_open':         len(open_),
        'overall_f1':     round(avg_f1(records) * 100, 2),
        'overall_recall': round(avg_recall(records) * 100, 2),
        'closed_acc':     round(accuracy(closed, True) * 100, 2),
        'open_acc':       round(accuracy(open_, False) * 100, 2),
        'bleu':           round(avg_bleu(records), 2),
    }

print("Scoring functions defined.")

Scoring functions defined.


In [17]:
# Score the downloaded JSONL files
import os

output_dir = os.path.expanduser('~/vlm_benchmark/outputs')

for fname in sorted(os.listdir(output_dir)):
    if fname.endswith('.jsonl'):
        path   = os.path.join(output_dir, fname)
        scores = score_jsonl(path)
        print(scores)
        print()

{'dataset': 'google_gemma-3-4b-it__slake.jsonl', 'n_total': 1061, 'n_closed': 416, 'n_open': 645, 'overall_f1': 17.62, 'overall_recall': 43.28, 'closed_acc': 64.66, 'open_acc': 24.65, 'bleu': 7.65}

{'dataset': 'google_medgemma-4b-it__slake.jsonl', 'n_total': 1061, 'n_closed': 416, 'n_open': 645, 'overall_f1': 55.95, 'overall_recall': 61.86, 'closed_acc': 76.68, 'open_acc': 47.29, 'bleu': 50.53}

{'dataset': 'google_medgemma-4b-it__vqa_rad.jsonl', 'n_total': 451, 'n_closed': 251, 'n_open': 200, 'overall_f1': 57.14, 'overall_recall': 62.53, 'closed_acc': 71.31, 'open_acc': 43.5, 'bleu': 44.44}



In [18]:
# 10 worst open-ended predictions (Gemma-3 on SLAKE):
import json, os

path = os.path.expanduser('~/vlm_benchmark/outputs/google_gemma-3-4b-it__slake.jsonl')
records = [json.loads(l) for l in open(path)]
open_records = [r for r in records if not r['is_closed']]

# Show 10 open-ended samples with low F1
low_f1 = sorted(open_records, key=lambda r: token_f1(r['prediction'], r['ground_truth'])['f1'])[:10]

print("10 worst open-ended predictions (Gemma-3 on SLAKE):\n")
for r in low_f1:
    scores = token_f1(r['prediction'], r['ground_truth'])
    print(f"Q:    {r['question']}")
    print(f"GT:   {r['ground_truth']}")
    print(f"Pred: {r['prediction']}")
    print(f"F1:   {scores['f1']:.3f}")
    print()

10 worst open-ended predictions (Gemma-3 on SLAKE):

Q:    Which part of the body does this image belong to?
GT:   Chest
Pred: Chest/Thorax
F1:   0.000

Q:    What is the main organ in the image?
GT:   Lung, Spinal Cord
Pred: Lungs
F1:   0.000

Q:    What is the largest organ in the picture?
GT:   Lung
Pred: The heart.
F1:   0.000

Q:    What diseases are included in the picture?
GT:   Lung Cancer
Pred: Here's a concise answer based on the CT scan image:

**Pneumothorax**
F1:   0.000

Q:    What is the main organ in the image?
GT:   Lung
Pred: The lungs.
F1:   0.000

Q:    What diseases are included in the picture?
GT:   Lung Cancer
Pred: Here's a concise answer based on the CT scan image:

**Mediastinal mass,
F1:   0.000

Q:    What is the main organ in the image?
GT:   Lung
Pred: Lungs
F1:   0.000

Q:    What is the largest organ in the picture?
GT:   Lung
Pred: The lungs.
F1:   0.000

Q:    What diseases are included in the picture?
GT:   Lung Cancer
Pred: Here's a concise answer ba

In [19]:
# Quantify how many Gemma-3 "wrong" answers are semantically close but surface-different
import json, os

path = os.path.expanduser('~/vlm_benchmark/outputs/google_gemma-3-4b-it__slake.jsonl')
records = [json.loads(l) for l in open(path)]
open_records = [r for r in records if not r['is_closed']]

zero_f1 = [r for r in open_records 
           if token_f1(r['prediction'], r['ground_truth'])['f1'] == 0.0]

# Among zero-F1 cases, how many predictions are non-empty (model tried but surface mismatch)
non_empty = [r for r in zero_f1 if r['prediction'].strip() != '']
empty     = [r for r in zero_f1 if r['prediction'].strip() == '']

print(f"Open-ended total:          {len(open_records)}")
print(f"Zero F1:                   {len(zero_f1)} ({len(zero_f1)/len(open_records)*100:.1f}%)")
print(f"  → Non-empty (tried):     {len(non_empty)} ({len(non_empty)/len(zero_f1)*100:.1f}% of zero-F1)")
print(f"  → Empty (no answer):     {len(empty)}")
print(f"\nImplication: model produced an answer but token mismatch")
print(f"penalized {len(non_empty)/len(open_records)*100:.1f}% of all open questions")

Open-ended total:          645
Zero F1:                   410 (63.6%)
  → Non-empty (tried):     410 (100.0% of zero-F1)
  → Empty (no answer):     0

Implication: model produced an answer but token mismatch
penalized 63.6% of all open questions


In [20]:
# Save this finding to a notes file now.
import json, os

finding = {
    "model": "google/gemma-3-4b-it",
    "dataset": "SLAKE",
    "finding": "surface_form_bias",
    "open_total": 645,
    "zero_f1_count": 410,
    "zero_f1_pct": 63.6,
    "non_empty_in_zero_f1": 410,
    "non_empty_pct": 100.0,
    "interpretation": (
        "All zero-F1 open-ended predictions were non-empty. "
        "Token-level F1 penalizes surface-form variation (Lungs vs Lung, "
        "Radiography vs X-Ray) without capturing semantic correctness. "
        "General VLMs not fine-tuned on dataset vocabulary are systematically "
        "disadvantaged relative to medical VLMs trained on these exact labels."
    )
}

notes_path = os.path.expanduser('~/vlm_benchmark/outputs/findings.json')
with open(notes_path, 'w') as f:
    json.dump([finding], f, indent=2)

print(f"Saved to {notes_path}")

Saved to /Users/shriyanshraj/vlm_benchmark/outputs/findings.json


In [ ]:
'''Two additional metrics alongside token-F1:
BERTScore: embedding-based similarity. "Lungs" and "Lung", "Radiography" and "X-Ray" will score very high because they are semantically close in embedding space. Fix for the surface-form problem.
Soft/Partial token match: normalize predictions before token matching by lowercasing, removing articles ("the", "a"), stripping plurals (simple stemming), and splitting slash-joined terms ("Chest/Thorax" → ["Chest", "Thorax"]).
''''

In [22]:
# Adding normalization to the tokenizer

import re, string
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

stemmer   = PorterStemmer()
STOPWORDS = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'this', 'that', 'of', 'in', 'for'}

def normalize_answer(text: str) -> str:
    """
    Normalize before tokenizing:
    1. Lowercase
    2. Strip markdown
    3. Take first sentence only
    4. Split slash-joined terms (Chest/Thorax → Chest Thorax)
    5. Remove stopwords
    6. Stem (Lungs → Lung, radiographs → radiograph)
    """
    # Markdown and punctuation
    text = re.sub(r'\*+', '', text)
    text = re.sub(r'\*\*.*?\*\*', lambda m: m.group().replace('**',''), text)
    # First sentence only
    text = re.split(r'(?<=[.!?])\s', text)[0]
    # Slash-joined terms
    text = text.replace('/', ' ')
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and stem
    tokens = [stemmer.stem(t) for t in tokens if t not in STOPWORDS]
    return ' '.join(tokens)

def token_f1_normalized(prediction: str, ground_truth: str) -> dict:
    """Token F1 with normalization applied to both prediction and ground truth."""
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens   = normalize_answer(ground_truth).split()

    if not pred_tokens or not gt_tokens:
        return {'f1': 0.0, 'precision': 0.0, 'recall': 0.0}

    pred_set = Counter(pred_tokens)
    gt_set   = Counter(gt_tokens)
    common   = sum((pred_set & gt_set).values())

    precision = common / len(pred_tokens)
    recall    = common / len(gt_tokens)
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)

    return {'f1': f1, 'precision': precision, 'recall': recall}

# Quick test on our known failure cases
test_cases = [
    ("Chest/Thorax",  "Chest"),
    ("Lungs",         "Lung"),
    ("The lungs.",    "Lung"),
    ("Radiography",   "X-Ray"),
    ("The heart.",    "Lung"),
]

print(f"{'Prediction':<25} {'GT':<15} {'F1_old':>8} {'F1_norm':>8}")
print("-" * 60)
for pred, gt in test_cases:
    old  = token_f1(pred, gt)['f1']
    new  = token_f1_normalized(pred, gt)['f1']
    print(f"{pred:<25} {gt:<15} {old:>8.3f} {new:>8.3f}")

Prediction                GT                F1_old  F1_norm
------------------------------------------------------------
Chest/Thorax              Chest              0.000    0.667
Lungs                     Lung               0.000    1.000
The lungs.                Lung               0.000    1.000
Radiography               X-Ray              0.000    0.000
The heart.                Lung               0.000    0.000


In [23]:
# Adding BERTScore as a metric
from bert_score import score as bert_score_fn
import torch

def compute_bertscore_batch(predictions: list, ground_truths: list) -> list:
    """
    Compute BERTScore F1 for a list of prediction/GT pairs.
    Returns a list of float scores (0-1).
    Uses distilbert-base-uncased for speed.
    """
    P, R, F = bert_score_fn(
        predictions,
        ground_truths,
        model_type='distilbert-base-uncased',
        lang='en',
        verbose=False,
        device='mps' if torch.backends.mps.is_available() else 'cpu'
    )
    return F.tolist()

# Quick test
preds = ["Chest/Thorax", "Lungs", "The lungs.", "Radiography", "The heart."]
gts   = ["Chest",        "Lung",  "Lung",        "X-Ray",       "Lung"]
scores = compute_bertscore_batch(preds, gts)

print(f"{'Prediction':<25} {'GT':<15} {'BERTScore F1':>12}")
print("-" * 55)
for pred, gt, s in zip(preds, gts, scores):
    print(f"{pred:<25} {gt:<15} {s:>12.3f}")

HTTP Error 504 thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Prediction                GT              BERTScore F1
-------------------------------------------------------
Chest/Thorax              Chest                  0.794
Lungs                     Lung                   0.911
The lungs.                Lung                   0.745
Radiography               X-Ray                  0.739
The heart.                Lung                   0.690


In [24]:
# Updated score_jsonl with all metrics
def score_jsonl_full(path: str) -> dict:
    records = [json.loads(l) for l in open(path)]
    records = [r for r in records if 'error' not in r]
    closed  = [r for r in records if r['is_closed']]
    open_   = [r for r in records if not r['is_closed']]

    def avg_metric(recs, fn):
        if not recs: return 0.0
        return sum(fn(r['prediction'], r['ground_truth']) for r in recs) / len(recs)

    def accuracy(recs, is_closed):
        if not recs: return 0.0
        return sum(is_correct(r['prediction'], r['ground_truth'], is_closed)
                   for r in recs) / len(recs)

    def accuracy_normalized(recs, is_closed):
        if not recs: return 0.0
        threshold = 0.5 if is_closed else 0.75
        return sum(
            1 for r in recs
            if token_f1_normalized(r['prediction'], r['ground_truth'])['recall'] >= threshold
        ) / len(recs)

    # BERTScore on open-ended only (closed is trivial yes/no)
    if open_:
        bert_scores = compute_bertscore_batch(
            [r['prediction']   for r in open_],
            [r['ground_truth'] for r in open_]
        )
        avg_bert = sum(bert_scores) / len(bert_scores)
    else:
        avg_bert = 0.0

    return {
        'dataset':          os.path.basename(path),
        'n_total':          len(records),
        'n_closed':         len(closed),
        'n_open':           len(open_),
        # Original metrics
        'overall_f1':       round(avg_metric(records, lambda p,g: token_f1(p,g)['f1']) * 100, 2),
        'overall_recall':   round(avg_metric(records, lambda p,g: token_f1(p,g)['recall']) * 100, 2),
        'closed_acc':       round(accuracy(closed, True) * 100, 2),
        'open_acc':         round(accuracy(open_, False) * 100, 2),
        'bleu':             round(avg_metric(records, compute_bleu), 2),
        # New metrics
        'overall_f1_norm':  round(avg_metric(records, lambda p,g: token_f1_normalized(p,g)['f1']) * 100, 2),
        'closed_acc_norm':  round(accuracy_normalized(closed, True) * 100, 2),
        'open_acc_norm':    round(accuracy_normalized(open_, False) * 100, 2),
        'bertscore_open':   round(avg_bert * 100, 2),
    }

print("Full scoring function defined.")

Full scoring function defined.


In [25]:
# Re-score all existing files
output_dir = os.path.expanduser('~/vlm_benchmark/outputs')

for fname in sorted(os.listdir(output_dir)):
    if fname.endswith('.jsonl'):
        path   = os.path.join(output_dir, fname)
        scores = score_jsonl_full(path)
        print(scores)
        print()

{'dataset': 'google_gemma-3-4b-it__slake.jsonl', 'n_total': 1061, 'n_closed': 416, 'n_open': 645, 'overall_f1': 17.62, 'overall_recall': 43.28, 'closed_acc': 64.66, 'open_acc': 24.65, 'bleu': 7.65, 'overall_f1_norm': 39.79, 'closed_acc_norm': 62.98, 'open_acc_norm': 30.23, 'bertscore_open': 75.44}

{'dataset': 'google_medgemma-4b-it__slake.jsonl', 'n_total': 1061, 'n_closed': 416, 'n_open': 645, 'overall_f1': 55.95, 'overall_recall': 61.86, 'closed_acc': 76.68, 'open_acc': 47.29, 'bleu': 50.53, 'overall_f1_norm': 57.95, 'closed_acc_norm': 76.68, 'open_acc_norm': 49.46, 'bertscore_open': 83.43}

{'dataset': 'google_medgemma-4b-it__vqa_rad.jsonl', 'n_total': 451, 'n_closed': 251, 'n_open': 200, 'overall_f1': 57.14, 'overall_recall': 62.53, 'closed_acc': 71.31, 'open_acc': 43.5, 'bleu': 44.44, 'overall_f1_norm': 58.27, 'closed_acc_norm': 71.31, 'open_acc_norm': 44.0, 'bertscore_open': 80.34}



In [14]:
# Dry run (20 samples per dataset)

MODEL_NAME = "google/medgemma-4b-it"

path_slake  = run_dataset(slake_en_test,  **slake_fns,  dataset_name='slake',    model_name=MODEL_NAME, max_samples=20)
path_vqarad = run_dataset(vqarad_test,    **vqarad_fns, dataset_name='vqa_rad',  model_name=MODEL_NAME, max_samples=20)
path_pvqa   = run_dataset(pvqa_test,      **pvqa_fns,   dataset_name='path_vqa', model_name=MODEL_NAME, max_samples=20)

google/medgemma-4b-it | slake: 100%|██████████████████████████████████████████████| 20/20 [04:44<00:00, 14.23s/it]


Saved 20 results (0 errors) → /Users/shriyanshraj/vlm_benchmark/outputs/google_medgemma-4b-it__slake.jsonl


google/medgemma-4b-it | vqa_rad: 100%|████████████████████████████████████████████| 20/20 [04:42<00:00, 14.10s/it]


Saved 20 results (0 errors) → /Users/shriyanshraj/vlm_benchmark/outputs/google_medgemma-4b-it__vqa_rad.jsonl


google/medgemma-4b-it | path_vqa: 100%|███████████████████████████████████████████| 20/20 [04:53<00:00, 14.69s/it]

Saved 20 results (0 errors) → /Users/shriyanshraj/vlm_benchmark/outputs/google_medgemma-4b-it__path_vqa.jsonl


In [1]:
# Experiment to reduce time (Failed)
# Reduce max_new_tokens. Most VQA answers are under 10 tokens. Dropping from 100 to 20 should cut generation time by 60-70% (expected).

def run_inference(image: Image.Image, question: str, is_closed: bool) -> str:
    prompt_text = build_prompt(question, is_closed)
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": prompt_text},
            ]
        }
    ]
    
    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = processor(
        text=text,
        images=image,
        return_tensors="pt"
    ).to(device)
    
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=20,      # reduced from 100
            do_sample=False,
        )
    
    input_len = inputs["input_ids"].shape[-1]
    generated = output_ids[0][input_len:]
    decoded   = processor.decode(generated, skip_special_tokens=True).strip()
    return decoded

print("Updated inference function defined.")

NameError: name 'Image' is not defined

In [17]:
# Checking time taken for each sample after reducing max tokens 100 -> 20
import time

sample    = slake_en_test[0]
image     = load_slake_image(sample['img_name'])
question  = sample['question']
is_closed = sample['answer_type'] == 'CLOSED'

start = time.time()
pred  = run_inference(image, question, is_closed)
elapsed = time.time() - start

print(f"Time per sample: {elapsed:.2f}s")
print(f"Prediction: {pred}")
print(f"Estimated SLAKE full run:  {elapsed * 1061 / 3600:.1f} hours")
print(f"Estimated VQA-RAD full run: {elapsed * 451 / 3600:.1f} hours")
# It actually got slower, which means the bottleneck is not generation

Time per sample: 22.51s
Prediction: Computed tomography (CT)
Estimated SLAKE full run:  6.6 hours
Estimated VQA-RAD full run: 2.8 hours


In [18]:
# Diagnostic Cell — Where is the time going?
import time

sample    = slake_en_test[0]
image     = load_slake_image(sample['img_name'])
question  = sample['question']
is_closed = sample['answer_type'] == 'CLOSED'
prompt_text = build_prompt(question, is_closed)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": prompt_text},
        ]
    }
]

# Time 1: chat template
t0 = time.time()
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print(f"Chat template:     {time.time()-t0:.2f}s")

# Time 2: processor (image encoding happens here)
t0 = time.time()
inputs = processor(text=text, images=image, return_tensors="pt").to(device)
print(f"Processor/encode:  {time.time()-t0:.2f}s")
print(f"Input shape: {inputs['input_ids'].shape}")

# Time 3: first token (prefill)
t0 = time.time()
with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=1, do_sample=False)
print(f"Prefill (1 token): {time.time()-t0:.2f}s")

# Time 4: full generation
t0 = time.time()
with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=20, do_sample=False)
print(f"Generation 20tok:  {time.time()-t0:.2f}s")

Chat template:     0.00s
Processor/encode:  0.06s
Input shape: torch.Size([1, 287])
Prefill (1 token): 17.68s
Generation 20tok:  19.74s


In [19]:
# Diagnostic Cell — Warm vs cold timing
import time

sample    = slake_en_test[0]
image     = load_slake_image(sample['img_name'])
question  = sample['question']
is_closed = sample['answer_type'] == 'CLOSED'

# Run 5 consecutive samples and time each
print("Timing 5 consecutive inferences on the same sample:\n")
for i in range(5):
    t0   = time.time()
    pred = run_inference(image, question, is_closed)
    elapsed = time.time() - t0
    print(f"  Run {i+1}: {elapsed:.2f}s  |  {pred}")

Timing 5 consecutive inferences on the same sample:

  Run 1: 19.69s  |  Computed tomography (CT)
  Run 2: 13.52s  |  Computed tomography (CT)
  Run 3: 16.82s  |  Computed tomography (CT)
  Run 4: 18.02s  |  Computed tomography (CT)
  Run 5: 16.09s  |  Computed tomography (CT)


In [20]:
# Then time across different samples
print("\nTiming 5 different SLAKE samples:\n")
for i, sample in enumerate(slake_en_test.select(range(5))):
    image     = load_slake_image(sample['img_name'])
    is_closed = sample['answer_type'] == 'CLOSED'
    
    t0   = time.time()
    pred = run_inference(image, sample['question'], is_closed)
    elapsed = time.time() - t0
    print(f"  Sample {i+1}: {elapsed:.2f}s  |  Q: {sample['question'][:40]}  |  Pred: {pred}")


Timing 5 different SLAKE samples:

  Sample 1: 14.06s  |  Q: What modality is used to take this image  |  Pred: Computed tomography (CT)
  Sample 2: 14.98s  |  Q: Which part of the body does this image b  |  Pred: Chest
  Sample 3: 12.30s  |  Q: What is the main organ in the image?  |  Pred: Lung
  Sample 4: 12.81s  |  Q: What is the largest organ in the picture  |  Pred: Lung
  Sample 5: 11.91s  |  Q: Does the picture contain liver?  |  Pred: No


In [21]:
# Shifting to collab due to high inference time on MPS
colab_code = '''
# ── Imports ───────────────────────────────────────────────────────────
import os, json, re, string, time
import torch
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from PIL import Image
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText
from nltk.tokenize import word_tokenize
from sacrebleu.metrics import BLEU
from collections import Counter
from tqdm import tqdm

# ── Device ────────────────────────────────────────────────────────────
def get_device():
    if torch.cuda.is_available():    return "cuda"
    if torch.backends.mps.is_available(): return "mps"
    return "cpu"

device = get_device()
print(f"Device: {device}")

# ── Image utils ───────────────────────────────────────────────────────
def to_rgb(img):
    return img if img.mode == "RGB" else img.convert("RGB")

def load_slake_image(img_name, imgs_dir):
    return Image.open(os.path.join(imgs_dir, img_name)).convert("RGB")

# ── Prompt ────────────────────────────────────────────────────────────
def build_prompt(question, is_closed):
    if is_closed:
        return f"Answer the question with yes or no. {question}"
    return f"{question} Answer concisely in as few words as possible."

# ── Inference ─────────────────────────────────────────────────────────
def run_inference(model, processor, device, image, question, is_closed):
    prompt_text = build_prompt(question, is_closed)
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": prompt_text},
        ]
    }]
    text   = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=text, images=image, return_tensors="pt"
    ).to(device)
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    input_len = inputs["input_ids"].shape[-1]
    generated = output_ids[0][input_len:]
    return processor.decode(generated, skip_special_tokens=True).strip()

# ── Scoring ───────────────────────────────────────────────────────────
def tokenize_answer(text):
    text = re.sub(r"\\*+", "", text).lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    return word_tokenize(text)

def token_f1(prediction, ground_truth):
    pred_tokens = tokenize_answer(prediction)
    gt_tokens   = tokenize_answer(ground_truth)
    if not pred_tokens or not gt_tokens:
        return {"f1": 0.0, "precision": 0.0, "recall": 0.0}
    pred_set = Counter(pred_tokens)
    gt_set   = Counter(gt_tokens)
    common      = sum((pred_set & gt_set).values())
    precision   = common / len(pred_tokens)
    recall      = common / len(gt_tokens)
    f1          = (2 * precision * recall / (precision + recall)
                   if (precision + recall) > 0 else 0.0)
    return {"f1": f1, "precision": precision, "recall": recall}

def is_correct(prediction, ground_truth, is_closed):
    scores    = token_f1(prediction, ground_truth)
    threshold = 0.5 if is_closed else 0.75
    return scores["recall"] >= threshold

bleu_metric = BLEU(effective_order=True)
def compute_bleu(prediction, ground_truth):
    return bleu_metric.sentence_score(
        hypothesis=prediction.lower(),
        references=[ground_truth.lower()]
    ).score

def score_jsonl(path):
    records = [json.loads(l) for l in open(path)]
    records = [r for r in records if "error" not in r]
    closed  = [r for r in records if r["is_closed"]]
    open_   = [r for r in records if not r["is_closed"]]
    def avg_f1(recs):
        if not recs: return 0.0
        return sum(token_f1(r["prediction"], r["ground_truth"])["f1"]
                   for r in recs) / len(recs)
    def avg_recall(recs):
        if not recs: return 0.0
        return sum(token_f1(r["prediction"], r["ground_truth"])["recall"]
                   for r in recs) / len(recs)
    def accuracy(recs, is_closed):
        if not recs: return 0.0
        return sum(is_correct(r["prediction"], r["ground_truth"], is_closed)
                   for r in recs) / len(recs)
    def avg_bleu(recs):
        if not recs: return 0.0
        return sum(compute_bleu(r["prediction"], r["ground_truth"])
                   for r in recs) / len(recs)
    return {
        "dataset":        os.path.basename(path),
        "n_total":        len(records),
        "n_closed":       len(closed),
        "n_open":         len(open_),
        "overall_f1":     round(avg_f1(records) * 100, 2),
        "overall_recall": round(avg_recall(records) * 100, 2),
        "closed_acc":     round(accuracy(closed, True) * 100, 2),
        "open_acc":       round(accuracy(open_, False) * 100, 2),
        "bleu":           round(avg_bleu(records), 2),
    }

# ── Runner ────────────────────────────────────────────────────────────
def run_dataset(model, processor, device, samples,
                get_image_fn, get_question_fn,
                get_answer_fn, get_is_closed_fn,
                dataset_name, model_name,
                output_dir, max_samples=None):
    os.makedirs(output_dir, exist_ok=True)
    safe_model = model_name.replace("/", "_")
    out_path   = os.path.join(output_dir, f"{safe_model}__{dataset_name}.jsonl")
    if max_samples:
        samples = samples.select(range(max_samples))
    results, errors = [], 0
    for i, sample in enumerate(tqdm(samples, desc=f"{model_name} | {dataset_name}")):
        try:
            image     = get_image_fn(sample)
            question  = get_question_fn(sample)
            answer    = get_answer_fn(sample)
            is_closed = get_is_closed_fn(sample)
            prediction = run_inference(
                model, processor, device, image, question, is_closed
            )
            results.append({
                "idx": i, "question": question,
                "ground_truth": answer, "prediction": prediction,
                "is_closed": is_closed,
                "model": model_name, "dataset": dataset_name,
            })
        except Exception as e:
            errors += 1
            results.append({
                "idx": i, "question": "", "ground_truth": "",
                "prediction": "", "is_closed": False,
                "model": model_name, "dataset": dataset_name,
                "error": str(e),
            })
    with open(out_path, "w") as f:
        for r in results:
            f.write(json.dumps(r) + "\\n")
    print(f"Saved {len(results)} results ({errors} errors) → {out_path}")
    return out_path
'''

script_path = os.path.expanduser('~/vlm_benchmark/scripts/vlm_eval_core.py')
with open(script_path, 'w') as f:
    f.write(colab_code)

print(f"Saved to {script_path}")

Saved to /Users/shriyanshraj/vlm_benchmark/scripts/vlm_eval_core.py


In [22]:
colab_nb = {
 "nbformat": 4,
 "nbformat_minor": 0,
 "metadata": {
  "colab": {"provenance": []},
  "kernelspec": {"name": "python3", "display_name": "Python 3"},
  "accelerator": "GPU"
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# VLM Medical VQA Evaluation\n", "Run on a T4 GPU runtime in Colab."]
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 1 — Check GPU\n",
    "!nvidia-smi\n"
   ],
   "execution_count": None, "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 2 — Install dependencies\n",
    "!pip install -q transformers==4.51.3 accelerate datasets evaluate \\\n",
    "    sacrebleu nltk sentencepiece protobuf huggingface_hub pillow tqdm\n"
   ],
   "execution_count": None, "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 3 — Mount Google Drive (outputs will be saved here)\n",
    "from google.colab import drive\n",
    "drive.mount('/content/drive')\n",
    "import os\n",
    "OUTPUT_DIR = '/content/drive/MyDrive/vlm_benchmark/outputs'\n",
    "os.makedirs(OUTPUT_DIR, exist_ok=True)\n",
    "print('Output dir:', OUTPUT_DIR)\n"
   ],
   "execution_count": None, "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 4 — HuggingFace login\n",
    "# Add your HF token as a Colab Secret named HF_TOKEN\n",
    "# Secrets are under the key icon in the left sidebar\n",
    "from google.colab import userdata\n",
    "from huggingface_hub import login\n",
    "login(token=userdata.get('HF_TOKEN'))\n",
    "print('Logged in.')\n"
   ],
   "execution_count": None, "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 5 — Download SLAKE images\n",
    "import zipfile, subprocess\n",
    "SLAKE_IMGS_DIR = '/content/slake_imgs'\n",
    "os.makedirs(SLAKE_IMGS_DIR, exist_ok=True)\n",
    "url = 'https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip'\n",
    "print('Downloading SLAKE images...')\n",
    "subprocess.run(['curl', '-L', url, '-o', f'{SLAKE_IMGS_DIR}/imgs.zip'],\n",
    "               capture_output=True)\n",
    "with zipfile.ZipFile(f'{SLAKE_IMGS_DIR}/imgs.zip', 'r') as z:\n",
    "    z.extractall(SLAKE_IMGS_DIR)\n",
    "imgs = [f for r,d,files in os.walk(SLAKE_IMGS_DIR)\n",
    "        for f in files if f.endswith(('.jpg','.png'))]\n",
    "print(f'Extracted {len(imgs)} images.')\n"
   ],
   "execution_count": None, "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 6 — Paste vlm_eval_core.py contents here\n",
    "# (copy the full contents of ~/vlm_benchmark/scripts/vlm_eval_core.py)\n",
    "# --- PASTE BELOW ---\n",
    "\n"
   ],
   "execution_count": None, "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 7 — Load datasets\n",
    "from datasets import load_dataset\n",
    "\n",
    "print('Loading SLAKE...')\n",
    "slake         = load_dataset('BoKelvin/SLAKE')\n",
    "slake_en_test = slake['test'].filter(lambda x: x['q_lang'] == 'en')\n",
    "print(f'SLAKE EN test: {len(slake_en_test)}')\n",
    "\n",
    "print('Loading VQA-RAD...')\n",
    "vqarad      = load_dataset('flaviagiammarino/vqa-rad')\n",
    "vqarad_test = vqarad['test']\n",
    "print(f'VQA-RAD test: {len(vqarad_test)}')\n"
   ],
   "execution_count": None, "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 8 — CONFIG: set model to run\n",
    "# Change this to switch models\n",
    "MODEL_NAME = 'google/medgemma-4b-it'\n",
    "# MODEL_NAME = 'google/gemma-3-4b-it'\n",
    "# MODEL_NAME = 'llava-hf/llava-v1.6-mistral-7b-hf'\n",
    "# MODEL_NAME = 'microsoft/llava-med-v1.5-mistral-7b'\n",
    "# MODEL_NAME = 'FreedomIntelligence/HuatuoGPT-Vision-7B'\n"
   ],
   "execution_count": None, "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 9 — Load model\n",
    "import torch\n",
    "from transformers import AutoProcessor, AutoModelForImageTextToText\n",
    "\n",
    "device = 'cuda' if torch.cuda.is_available() else 'cpu'\n",
    "print(f'Device: {device}')\n",
    "\n",
    "dtype = torch.bfloat16 if device == 'cuda' else torch.float32\n",
    "\n",
    "print(f'Loading {MODEL_NAME}...')\n",
    "processor = AutoProcessor.from_pretrained(MODEL_NAME)\n",
    "model     = AutoModelForImageTextToText.from_pretrained(\n",
    "    MODEL_NAME,\n",
    "    torch_dtype=dtype,\n",
    "    device_map='auto',\n",
    ")\n",
    "model.eval()\n",
    "print('Model loaded.')\n"
   ],
   "execution_count": None, "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 10 — Warmup + speed check\n",
    "import time\n",
    "sample    = slake_en_test[0]\n",
    "image     = load_slake_image(sample['img_name'], SLAKE_IMGS_DIR + '/imgs')\n",
    "is_closed = sample['answer_type'] == 'CLOSED'\n",
    "\n",
    "# Warmup\n",
    "_ = run_inference(model, processor, device, image, sample['question'], is_closed)\n",
    "\n",
    "# Timed\n",
    "t0   = time.time()\n",
    "pred = run_inference(model, processor, device, image, sample['question'], is_closed)\n",
    "elapsed = time.time() - t0\n",
    "print(f'Time per sample: {elapsed:.2f}s')\n",
    "print(f'Predicted: {pred}')\n",
    "print(f'Estimated SLAKE full run:   {elapsed * 1061 / 3600:.1f} hrs')\n",
    "print(f'Estimated VQA-RAD full run: {elapsed * 451  / 3600:.1f} hrs')\n"
   ],
   "execution_count": None, "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 11 — Run SLAKE\n",
    "slake_path = run_dataset(\n",
    "    model, processor, device,\n",
    "    slake_en_test,\n",
    "    get_image_fn     = lambda s: load_slake_image(s['img_name'], SLAKE_IMGS_DIR + '/imgs'),\n",
    "    get_question_fn  = lambda s: s['question'],\n",
    "    get_answer_fn    = lambda s: s['answer'],\n",
    "    get_is_closed_fn = lambda s: s['answer_type'] == 'CLOSED',\n",
    "    dataset_name     = 'slake',\n",
    "    model_name       = MODEL_NAME,\n",
    "    output_dir       = OUTPUT_DIR,\n",
    ")\n",
    "print(score_jsonl(slake_path))\n"
   ],
   "execution_count": None, "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 12 — Run VQA-RAD\n",
    "vqarad_path = run_dataset(\n",
    "    model, processor, device,\n",
    "    vqarad_test,\n",
    "    get_image_fn     = lambda s: to_rgb(s['image']),\n",
    "    get_question_fn  = lambda s: s['question'],\n",
    "    get_answer_fn    = lambda s: s['answer'],\n",
    "    get_is_closed_fn = lambda s: s['answer'].strip().lower() in ('yes', 'no'),\n",
    "    dataset_name     = 'vqa_rad',\n",
    "    model_name       = MODEL_NAME,\n",
    "    output_dir       = OUTPUT_DIR,\n",
    ")\n",
    "print(score_jsonl(vqarad_path))\n"
   ],
   "execution_count": None, "outputs": []
  },
  {
   "cell_type": "code",
   "metadata": {},
   "source": [
    "# Cell 13 — Delete model from memory before loading next one\n",
    "import gc\n",
    "del model, processor\n",
    "gc.collect()\n",
    "torch.cuda.empty_cache()\n",
    "print('Memory cleared. Change MODEL_NAME in Cell 8 and re-run Cells 9-12.')\n"
   ],
   "execution_count": None, "outputs": []
  }
 ]
}

import json
nb_path = os.path.expanduser('~/vlm_benchmark/notebooks/colab_vqa_eval.ipynb')
with open(nb_path, 'w') as f:
    json.dump(colab_nb, f, indent=1)
print(f'Notebook saved to {nb_path}')

Notebook saved to /Users/shriyanshraj/vlm_benchmark/notebooks/colab_vqa_eval.ipynb
